# 🔒 Day 03a: Pydantic Models — Data Validation That Writes Itself

---

## 🎯 What You'll Master Today
- Pydantic V2 model creation and validation
- Field validators (before/after, custom logic)
- Model validators (cross-field validation)
- Nested models and complex schemas
- Serialization: model_dump, model_dump_json

---

## 🎭 Why Pydantic Matters

**Pydantic is the backbone of FastAPI.** Every request body, response, config, and DB schema goes through Pydantic. In interviews, you'll be asked "how does FastAPI validate data?" — the answer is always Pydantic.

```
╔═══════════════════════════════════════════════════════════════════╗
║  PYDANTIC V2 — THE ESSENTIALS                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  • Data validation using Python type hints                       ║
║  • V2 is rewritten in Rust → 5-50x faster than V1              ║
║  • Automatic serialization/deserialization                       ║
║  • JSON Schema generation → FastAPI docs                        ║
║  • Field-level and model-level validators                        ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# Basic Pydantic Model — Write This From Memory
# ============================================================
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Optional
from datetime import datetime

class User(BaseModel):
    name: str = Field(..., min_length=2, max_length=50)
    email: str
    age: int = Field(..., ge=1, le=150)
    bio: Optional[str] = None
    is_active: bool = True
    created_at: datetime = Field(default_factory=datetime.utcnow)

# Valid user
user = User(name="Sarthak", email="s@mail.com", age=25)
print(f"User: {user}")
print(f"Dict: {user.model_dump()}")
print(f"JSON: {user.model_dump_json(indent=2)}")

In [ ]:
# ============================================================
# Validation in Action — Pydantic catches errors
# ============================================================
from pydantic import ValidationError

# Let's try invalid data
test_cases = [
    {"name": "A", "email": "s@mail.com", "age": 25},    # name too short
    {"name": "Sarthak", "email": "s@mail.com", "age": -5}, # negative age
    {"name": "Sarthak", "email": "s@mail.com", "age": 200}, # age > 150
    {"name": "Sarthak"},  # missing required fields
]

for i, data in enumerate(test_cases):
    try:
        user = User(**data)
        print(f"Test {i+1}: ✅ Valid")
    except ValidationError as e:
        print(f"Test {i+1}: ❌ {e.error_count()} error(s)")
        for err in e.errors():
            print(f"   Field: {err['loc']}, Error: {err['msg']}")

In [ ]:
# ============================================================
# Field Validators — Custom validation logic
# ============================================================
import re

class UserWithValidation(BaseModel):
    name: str
    email: str
    password: str = Field(..., min_length=8)
    age: int
    
    @field_validator('email')
    @classmethod
    def validate_email(cls, v):
        """Validate email format."""
        pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
        if not re.match(pattern, v):
            raise ValueError('Invalid email format')
        return v.lower()  # normalize to lowercase
    
    @field_validator('name')
    @classmethod
    def validate_name(cls, v):
        """Name must be alphabetic."""
        if not v.replace(' ', '').isalpha():
            raise ValueError('Name must contain only letters')
        return v.strip().title()  # normalize: "  john doe " → "John Doe"
    
    @field_validator('password')
    @classmethod
    def validate_password(cls, v):
        """Password must have uppercase, lowercase, digit."""
        if not re.search(r'[A-Z]', v):
            raise ValueError('Password must contain an uppercase letter')
        if not re.search(r'[a-z]', v):
            raise ValueError('Password must contain a lowercase letter')
        if not re.search(r'\d', v):
            raise ValueError('Password must contain a digit')
        return v

# Valid
user = UserWithValidation(
    name="  sarthak  ", email="S@MAIL.COM", password="Secure123", age=25
)
print(f"Name normalized: '{user.name}'")
print(f"Email normalized: '{user.email}'")

# Invalid password
try:
    UserWithValidation(name="Test", email="t@m.com", password="weak", age=25)
except ValidationError as e:
    for err in e.errors():
        print(f"❌ {err['loc']}: {err['msg']}")

In [ ]:
# ============================================================
# Model Validator — Cross-field validation
# ============================================================

class DateRange(BaseModel):
    start_date: datetime
    end_date: datetime
    
    @model_validator(mode='after')
    def validate_dates(self):
        """end_date must be after start_date."""
        if self.end_date <= self.start_date:
            raise ValueError('end_date must be after start_date')
        return self

class PasswordChange(BaseModel):
    password: str
    confirm_password: str
    
    @model_validator(mode='after')
    def passwords_match(self):
        if self.password != self.confirm_password:
            raise ValueError('Passwords do not match')
        return self

# Valid
dr = DateRange(start_date="2024-01-01", end_date="2024-12-31")
print(f"✅ Date range: {dr.start_date.date()} → {dr.end_date.date()}")

# Invalid
try:
    DateRange(start_date="2024-12-31", end_date="2024-01-01")
except ValidationError as e:
    print(f"❌ {e.errors()[0]['msg']}")

try:
    PasswordChange(password="Secure123", confirm_password="Different456")
except ValidationError as e:
    print(f"❌ {e.errors()[0]['msg']}")

In [ ]:
# ============================================================
# Nested Models — Complex schemas
# ============================================================

class Address(BaseModel):
    street: str
    city: str
    state: str
    zip_code: str = Field(..., pattern=r'^\d{5}(-\d{4})?$')  # US zip

class Company(BaseModel):
    name: str
    address: Address  # Nested model!

class Employee(BaseModel):
    name: str
    email: str
    company: Company  # Nested 2 levels deep!
    skills: list[str] = []
    metadata: dict[str, str] = {}  # key-value pairs

# Pydantic handles nested validation automatically
emp = Employee(
    name="Sarthak",
    email="s@mail.com",
    company={
        "name": "TechCorp",
        "address": {
            "street": "123 Main St",
            "city": "San Francisco",
            "state": "CA",
            "zip_code": "94102"
        }
    },
    skills=["Python", "FastAPI", "Docker"],
    metadata={"team": "backend", "level": "senior"}
)

print(f"Employee: {emp.name}")
print(f"Company: {emp.company.name}")
print(f"City: {emp.company.address.city}")
print(f"\nJSON Schema preview:")
import json
print(json.dumps(Employee.model_json_schema(), indent=2)[:500])

In [ ]:
# ============================================================
# Serialization Patterns — model_dump options
# ============================================================

class UserFull(BaseModel):
    id: int
    name: str
    email: str
    password_hash: str  # should NEVER be in response!
    is_active: bool = True
    created_at: datetime = Field(default_factory=datetime.utcnow)

user = UserFull(id=1, name="Sarthak", email="s@m.com", password_hash="$2b$12...")

# Default: everything
print("Full dump:")
print(user.model_dump())

# Exclude sensitive fields
print("\nExclude password:")
print(user.model_dump(exclude={'password_hash'}))

# Include only specific fields
print("\nInclude only id, name:")
print(user.model_dump(include={'id', 'name'}))

# Exclude None values
print("\nExclude None:")
print(user.model_dump(exclude_none=True))

# By alias (if you have field aliases)
# print(user.model_dump(by_alias=True))

In [ ]:
# ============================================================
# Separate Create/Read/Update Schemas — The Pattern
# ============================================================
# This is the #1 pattern in FastAPI production apps

class UserBase(BaseModel):
    """Shared fields."""
    name: str = Field(..., min_length=2)
    email: str

class UserCreate(UserBase):
    """What client sends to create."""
    password: str = Field(..., min_length=8)

class UserUpdate(BaseModel):
    """What client sends to update (all optional)."""
    name: Optional[str] = Field(None, min_length=2)
    email: Optional[str] = None

class UserInDB(UserBase):
    """What's stored in database."""
    id: int
    password_hash: str
    is_active: bool = True

class UserResponse(UserBase):
    """What client receives back (NO password!)."""
    id: int
    is_active: bool

# Usage:
# POST /users  → receives UserCreate  → returns UserResponse
# GET  /users  → returns list[UserResponse]
# PATCH /users → receives UserUpdate  → returns UserResponse

print("Schema inheritance pattern:")
print("UserBase → UserCreate (+ password)")
print("UserBase → UserResponse (+ id, is_active)")
print("UserBase → UserInDB (+ id, password_hash, is_active)")
print("\nUserUpdate has ALL fields optional for PATCH")

---

## 🗺️ Pydantic Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  PYDANTIC V2 QUICK REFERENCE                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  MODEL:     class User(BaseModel): name: str                     ║
║  FIELD:     Field(..., min_length=2, ge=1, pattern=r'...')       ║
║  OPTIONAL:  field: Optional[str] = None                          ║
║                                                                   ║
║  VALIDATOR: @field_validator('field') + @classmethod             ║
║  MODEL VAL: @model_validator(mode='after')                       ║
║                                                                   ║
║  SERIALIZE: model_dump(), model_dump_json()                      ║
║  OPTIONS:   exclude, include, exclude_none, exclude_unset        ║
║                                                                   ║
║  PATTERN:   Base → Create (+ password)                           ║
║             Base → Response (+ id, no password)                  ║
║             Update (all Optional)                                  ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What is Pydantic? | Data validation library using Python type hints. Backbone of FastAPI |
| 2 | V1 vs V2? | V2 rewritten in Rust (5-50x faster), model_dump instead of dict() |
| 3 | How to validate email? | @field_validator with regex or use pydantic[email] EmailStr |
| 4 | Cross-field validation? | @model_validator(mode='after') — access all fields |
| 5 | Create vs Update schema? | Create: required fields + password. Update: all Optional |
| 6 | How to hide password in response? | Separate UserResponse model without password field |
| 7 | model_dump vs model_dump_json? | model_dump → dict. model_dump_json → JSON string |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Pydantic V2 validates using type hints (Rust core)   │
## │  • Field() for constraints: min/max, regex, ge/le       │
## │  • @field_validator for custom per-field logic           │
## │  • @model_validator for cross-field validation           │
## │  • Separate schemas: Create, Update, Response, InDB     │
## │  • model_dump(exclude={'password'}) for safe responses  │
## │  • Nested models validate automatically                  │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Dependency Injection** — FastAPI's secret weapon for clean architecture